In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [ ]:
import pandas as pd
import re
import nltk

# Download the 'stopwords' package, which contains common words (is, the, at) to be filtered out
nltk.download('stopwords')
# Download 'wordnet', a lexical database used by the Lemmatizer to find the base form of words
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
def lower_order(text):
    return text.lower()

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def removeunwanted_characters(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    # Initialize the tools for normalization
    wordnet = WordNetLemmatizer()
    porter = PorterStemmer()

    # Convert the input to small/lower order.
    data = lower_order(dataset)
    # Remove URLs
    data = remove_urls(data)
    # Remove emojis
    data = remove_emoji(data)
    # Remove all other unwanted characters.
    data = removeunwanted_characters(data)
    # Create tokens.
    tokens = data.split()
    # Remove stopwords:
    tokens = [word for word in tokens if word not in stop_words]

    if rule == "lemmatize":
        tokens = [wordnet.lemmatize(word, pos='v') for word in tokens]
    elif rule == "stem":
        tokens = [porter.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [ ]:
# The 'ISO-8859-1' encoding is used to handle special characters common in social media text
data = pd.read_csv("/content/drive/MyDrive/AI_ML/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")

# Apply the cleaning pipeline to the 'text' column
# .dropna() ensures we don't process empty rows, which would cause the pipeline to crash
# .apply(lambda x: ...) runs every single tweet through your text_cleaning_pipeline function
cleaned_tokens = data["text"].dropna().apply(lambda x: text_cleaning_pipeline(x))

In [ ]:
data.head(10)

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0
5,"Alec Baldwin returns as Trump on SNL, where an...",0
6,REPUBLICANS- YOU OWN YOUR TRUMP!!,0
7,RT @AnsarAAbbasi: For #Trump #Obama #Bush .......,0
8,A federal appeals court has refused to immedia...,0
9,Court denies Trump request to immediately rest...,0


In [ ]:
# Import the utility to split the dataset into training and testing subsets
from sklearn.model_selection import train_test_split

# Split the 'cleaned_tokens' (features) and the 'Sentiment' (target labels)
# we use .loc[cleaned_tokens.index] to ensure labels match the rows we didn't drop
# test_size=0.2 means 80% of data trains the model, while 20% is saved to test it
# random_state=42 ensures the split is reproducible every time the code runs
X_train, X_test, y_train, y_test = train_test_split(
    cleaned_tokens,
    data["Sentiment"].loc[cleaned_tokens.index],
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Initialize the vectorizer
tfidf = TfidfVectorizer()
# fit_transform learns the vocabulary and scales the training data
X_train_tfidf = tfidf.fit_transform(X_train)
# transform uses the training vocabulary to scale the test data
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
# Import Logistic Regression for classification and classification_report for metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize the Logistic Regression model
model = LogisticRegression()

# Train (fit) the model using the numerical training features and their known labels
model.fit(X_train_tfidf, y_train)

# Use the trained model to predict sentiments for the unseen test data
y_pred = model.predict(X_test_tfidf)

# Print the final performance metrics: Precision, Recall, F1-Score, and Accuracy
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.96    248563
           1       0.92      0.90      0.91    121462

    accuracy                           0.94    370025
   macro avg       0.94      0.93      0.93    370025
weighted avg       0.94      0.94      0.94    370025

